## Ground Truth Priors

In [ ]:
# Filepaths
# name: filepath
# TODO: Adapt with real filepaths
BASE = "outputs/new_experiments_try_4_presentation"
# MODELS = {
#     "Baseline": f"{BASE}/lego_baseline_no_prior",
#     "Baseline + direct": f"{BASE}/lego_photometric_warmup_direct",
#     "Baseline + zncc": f"{BASE}/lego_photometric_warmup_zncc",
#     "Albedo + zncc": f"{BASE}/lego_albedo_warmup_zncc"
# }
MODELS = {
    "Baseline": f"{BASE}/lego_baseline_no_prior",
    "With GT": f"{BASE}/lego_albedo_warmup_zncc",
    "With Diffusion Priors": f"{BASE}/lego_diffusion_zncc"
}

In [ ]:
METRICS = MODELS.copy()
for x in METRICS:
    METRICS[x] += "/metrics_log.json"

In [ ]:
METRICS

In [ ]:
import json

METRICS_JSON = {}

for metric_key in METRICS:
    with open(f"{METRICS[metric_key]}") as metrics_json_file:
        METRICS_JSON[metric_key] = json.load(metrics_json_file)


In [ ]:
metrics_60k_iteration = {}
for metric_key in METRICS_JSON:
    metrics_60k_iteration[metric_key] = METRICS_JSON[metric_key][-1]


In [ ]:
METRICS_TO_SHOW_KEYS = {"test_psnr", "test_albedo_psnr_aligned", "test_normal_ang_err"}
filtered_metrics_60k_iteration = {}
for model in metrics_60k_iteration:
    model_metrics = metrics_60k_iteration[model]
    filtered_metrics_60k_iteration[model] = {key: model_metrics[key] for key in model_metrics.keys() & METRICS_TO_SHOW_KEYS}
# filtered_metrics_60k_iteration


In [ ]:
filtered_metrics_60k_iteration

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
RELIGHTING_METRICS = MODELS.copy()
for x in METRICS:
    RELIGHTING_METRICS[x] += "/all_relighting/relighting_metrics.json"

In [ ]:
RELIGHTING_METRICS

In [ ]:
import json

RELIGHTING_JSON = {}

for metric_key in RELIGHTING_METRICS:
    with open(f"{RELIGHTING_METRICS[metric_key]}") as metrics_json_file:
        RELIGHTING_JSON[metric_key] = json.load(metrics_json_file)

In [ ]:
import numpy as np

SELECTED_SCENES = {"city", "courtyard", "snow"}
KEY = "scale_invariant_psnr"
def calculate_mean_psnr(data, scenes_to_include=SELECTED_SCENES):
    mean_psnr = {}
    for model_name, scenes in data.items():
        psnr_values = [
            metrics[KEY]
            for scene_name, metrics in scenes.items()
            if scene_name in scenes_to_include
            and metrics[KEY] is not None
        ]
        mean_psnr[model_name] = np.mean(psnr_values) if psnr_values else None
    return mean_psnr

mean_psnr_by_model = calculate_mean_psnr(RELIGHTING_JSON)
print(mean_psnr_by_model)

## This is the Script for creating the diagram

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

models = list(filtered_metrics_60k_iteration.keys())

color_map = {
    'Baseline': '#E83A6E',
    'Baseline + direct': '#4472C4',
    'Baseline + zncc': '#70AD47',
    'Albedo + zncc': '#9B59B6',
}
cmap = cm.get_cmap('tab10')
colors = [color_map.get(model, cmap(i % 10)) for i, model in enumerate(models)]

metric_labels = ['Novel-View Synthesis PSNR', 'Mean Relighting PSNR (Scale aligned)']

def get_vals(model):
    return [
        filtered_metrics_60k_iteration[model]['test_psnr'],
        mean_psnr_by_model[model],
        # filtered_metrics_60k_iteration[model]['test_albedo_psnr_aligned'],
    ]

n_metrics = 2
n_models = len(models)
bar_width = 0.2        # visual width of each bar
bar_step = 0.22         # spacing between bar centers within a group
x = np.arange(n_metrics)

y_max = max(
    v for model in models for v in get_vals(model) if v is not None
) * 1.25

for n_visible in range(1, n_models + 1):
    fig, ax = plt.subplots(figsize=(12, 5))

    for i, model in enumerate(models):
        offset = (i - (n_models - 1) / 2) * bar_step
        if i < n_visible:
            vals = get_vals(model)
            bars = ax.bar(x + offset, vals, bar_width, color=colors[i], label=model, zorder=3)

            for bar, val in zip(bars, vals):
                if val is not None:
                    ax.text(
                        bar.get_x() + bar.get_width() / 2,
                        val + y_max * 0.01,
                        f'{val:.1f}',
                        ha='center', va='bottom',
                        fontsize=9, color='#333333'
                    )

    ax.set_xticks(x)
    ax.set_xticklabels(metric_labels, fontsize=14)
    ax.set_ylabel('Value (PSNR)', fontsize=14)
    ax.set_ylim(0, y_max)
    ax.set_xlim(-0.6, n_metrics - 0.4)
    ax.yaxis.grid(True, color='#e1e0d9', linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#c3c2b7')
    ax.spines['bottom'].set_color('#c3c2b7')
    ax.tick_params(colors='#52514e', labelsize=13)
    ax.legend(fontsize=11, frameon=True, loc='upper right')
    plt.tight_layout()
    plt.savefig(f'outputs/comparison_{n_visible}models.pdf', dpi=150, bbox_inches='tight')
    plt.savefig(f"outputs/comparison_{n_visible}models.png")
    plt.show()

## Relighting Metrics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

# Assuming RELIGHTING_JSON is already defined
data = RELIGHTING_JSON
scenes = []
for method, method_data in data.items():
    for scene in method_data:
        if scene not in scenes:
            scenes.append(scene)
 
methods = list(data.keys())
colors = {'Baseline': '#4C72B0', 'With GT': '#55A868', 'With Diffusion Priors': '#C44E52'}

for scene in scenes:
    # Gather methods that have data for this scene
    present_methods = [m for m in methods if scene in data[m] and data[m][scene].get('scale_invariant_psnr') is not None]
 
    if not present_methods:
        continue
 
    fig, ax = plt.subplots(figsize=(7, 5))
 
    # We are grouping by metric now (X-axis ticks)
    metrics = ['Raw PSNR', 'Scale-invariant PSNR']
    x = np.arange(len(metrics))
    
    n_methods = len(present_methods)
    total_width = 0.6  # Total width for all bars within a single metric group
    width = total_width / n_methods
 
    for i, m in enumerate(present_methods):
        entry = data[m][scene]
        raw_val = entry.get('raw_psnr')
        si_val = entry.get('scale_invariant_psnr')
 
        # Using np.nan for missing raw values so matplotlib skips plotting them
        y_vals = [
            raw_val if raw_val is not None else np.nan,
            si_val if si_val is not None else np.nan
        ]
 
        # Calculate X position offset for this specific method's bar
        offset = (i - n_methods / 2 + 0.5) * width
        x_pos = x + offset
 
        # Fetch color from your dict, fallback to standard colormap if not found
        c = colors.get(m, plt.cm.tab10(i % 10))
 
        bars = ax.bar(x_pos, y_vals, width, label=m, color=c, edgecolor='black')
 
        # Add value text on top of the bars
        for bx, by in zip(x_pos, y_vals):
            if not np.isnan(by):
                ax.text(bx, by + 0.3, f'{by:.2f}', ha='center', va='bottom', fontsize=9)
 
    # Formatting
    ax.set_xticks(x)
    ax.set_xticklabels(metrics, fontsize=11, fontweight='bold')
    ax.set_title(f'PSNR per Metric - {scene}', fontsize=14, fontweight='bold')
    ax.set_ylabel('PSNR (dB)')
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    
    # Legend formatting - placed below the chart to keep the metrics clear
    ax.legend(title='Models', fontsize=9, loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=len(present_methods))
 
    fig.tight_layout()
    
    # Clean the scene name to ensure it's a valid filesystem path
    safe_scene_name = "".join([c if c.isalnum() else "_" for c in scene])
    
    # fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show() # Crucial: close the figure to free up memory on each loop iteration

In [ ]:
data_to_analyze = RELIGHTING_JSON

In [ ]:
data_to_analyze

In [ ]:
import pandas as pd
# Create a list to hold flattened records
records = []

for model_name, datasets in data_to_analyze.items():
    for dataset_name, metrics in datasets.items():
        # Initialize a dictionary for the row with our primary keys
        row = {'Model': model_name, 'Environment': dataset_name}
        # Merge the metrics into this row
        row.update(metrics)
        records.append(row)

# Create the DataFrame
df_flat = pd.DataFrame(records)

# Display the first few rows
print(df_flat.head())

In [ ]:
df_flat

In [ ]:
datasets_to_keep = ["city", "courtyard", "fireplace", "forest", "snow", "night"]
df_filtered = df_flat[df_flat["Environment"].isin(datasets_to_keep)]

In [ ]:
df_filtered

In [ ]:
cols_to_remove = ["scale_factor", "scale_invariant_ssim", "scale_invariant_mse", "raw_ssim", "raw_mse"]
df_filtered2 = df_filtered.drop(columns=cols_to_remove)

In [ ]:
df_filtered2

In [ ]:
df_comparison_scale_invariant = df_flat.pivot(
    index='Environment', 
    columns='Model', 
    values='scale_invariant_psnr'
).dropna()

df_comparison_raw = df_flat.pivot(
    index='Environment', 
    columns='Model', 
    values='raw_psnr'
).dropna()

In [ ]:
df_comparison_raw

In [ ]:
df_comparison_raw.style.highlight_max(axis=1, color='darkgoldenrod')

In [ ]:
# 1. Calculate the delta for Diffusion Priors vs Baseline
df_comparison_raw['Delta: Diffusion Priors'] = (
    df_comparison_raw['With Diffusion Priors'] - df_comparison_raw['Baseline']
)

# 2. Calculate the delta for GT vs Baseline
df_comparison_raw['Delta: GT'] = (
    df_comparison_raw['With GT'] - df_comparison_raw['Baseline']
)

In [ ]:
# 3. Define the red/green formatting function
def color_deltas(value):
    if pd.isna(value):
        return ''
    # Green if positive (better), red if negative (worse)
    color = 'green' if value > 0 else 'red'
    return f'color: {color}; font-weight: bold;'

# 4. Apply the style ONLY to the delta columns
delta_columns = ['Delta: Diffusion Priors', 'Delta: GT']

# Note: Use .map() for pandas 2.1+, or .applymap() for older pandas versions
styled_df = (
    df_comparison_raw.style
    .map(color_deltas, subset=delta_columns)
    # Format all numbers to 2 decimal places for a clean look
    .format(precision=2) 
)

# Display the styled table
styled_df

In [ ]:
styled_df.highlight_max(axis=1, color='darkgoldenrod')

## Material Evaluation

In [ ]:
MATERIAL_METRIC = MODELS.copy()
for x in MATERIAL_METRIC:
    MATERIAL_METRIC[x] += "/material_evaluation_metrics.json"

In [ ]:
import json

MATERIAL_METRICS_JSON = {}

for metric_key in METRICS:
    with open(f"{MATERIAL_METRIC[metric_key]}") as metrics_json_file:
        MATERIAL_METRICS_JSON[metric_key] = json.load(metrics_json_file)

In [ ]:
MATERIAL_METRICS_JSON

In [ ]:
data = {}
for model in MATERIAL_METRICS_JSON:
    values = MATERIAL_METRICS_JSON[model]
    data[model] = {
        "Mean Albedo PSNR (aligned)": values["albedo"]["mean_aligned_psnr"],
        "Mean Normal Angular Error": values["normal"]["mean_angular_error_deg"]
    }


In [ ]:
data
df = pd.DataFrame(data)
df.transpose()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 1. Update typography to a clean sans-serif matching typical poster fonts
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']

# Assuming 'data' is defined beforehand
# df = pd.DataFrame(data)
# df = df.reindex(sorted(df.columns), axis=1)

models = list(df.columns)

# ── Reorder so the Diffusion Priors model sits in the middle ──
diffusion_keys = {'with diffusion priors', 'with difusion priors'}  # covers typo variant
diffusion_models = [m for m in models if m.lower() in diffusion_keys]
other_models     = [m for m in models if m.lower() not in diffusion_keys]

if diffusion_models:
    mid = len(other_models) // 2
    models = other_models[:mid] + diffusion_models + other_models[mid:]
# ─────────────────────────────────────────────────────────────
metrics = list(df.index)

# 2. Refine color map to match the TUM poster theme
color_map = {
    'Baseline': '#999999',              # Neutral Gray
    'Baseline + direct': '#4472C4',
    'Baseline + zncc': '#70AD47',
    'Albedo + zncc': '#9B59B6',
    
    # Add capitalized versions to match your DataFrame columns exactly
    'With GT': '#333333',               
    'With Diffusion Priors': '#0065BD', 
    
    # Keeping lowercase versions just in case
    'with GT': '#333333',               
    'with Diffusion Priors': '#0065BD', 
    'with Difusion Priors': '#0065BD'   
}

# Apply mapped colors with a fallback
colors = [color_map.get(model, '#A6A6A6') for model in models]

# Apply mapped colors with a fallback
colors = [color_map.get(model, '#A6A6A6') for model in models]

n_models = len(models)
bar_width = 0.2        
bar_step = 0.22         

# 3. Modify styling function to remove legend and accept dynamic x-ticks
def apply_styling(ax, y_max, metric_name, models, offsets):
    # Set the x-ticks directly under each bar
    ax.set_xticks(offsets)
    
    # Format labels (e.g., adding linebreaks for longer names to save space)
    formatted_labels = [m.replace('with ', 'With\n') if 'with' in m.lower() else m for m in models]
    ax.set_xticklabels(formatted_labels, fontsize=12)
    
    if 'PSNR' in metric_name:
        ax.set_ylabel('Value (PSNR)', fontsize=14)
        ax.set_title(f'{metric_name}\n(Higher is Better)', pad=15)
    else:
        ax.set_ylabel('Angular Error', fontsize=14)
        ax.set_title(f'{metric_name}\n(Lower is Better)', pad=15)
        
    ax.set_ylim(0, y_max)
    
    # Tighten x-limits dynamically based on the number of bars
    ax.set_xlim(offsets[0] - bar_step, offsets[-1] + bar_step) 
    
    ax.yaxis.grid(True, color='#e1e0d9', linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#c3c2b7')
    ax.spines['bottom'].set_color('#c3c2b7')
    ax.tick_params(colors='#52514e', labelsize=12)

# --- Plot 1: PSNR ---
metric1 = 'Mean Albedo PSNR (aligned)'
if metric1 in df.index:
    fig1, ax1 = plt.subplots(figsize=(8, 6))
    y_max1 = df.loc[metric1].max() * 1.25
    offsets1 = []
    
    for i, model in enumerate(models):
        offset = (i - (n_models - 1) / 2) * bar_step
        offsets1.append(offset)
        val = df.loc[metric1, model]
        
        # Plot bar without label for legend
        bars = ax1.bar([offset], [val], bar_width, color=colors[i], zorder=3)
        for bar in bars:
            ax1.text(
                bar.get_x() + bar.get_width() / 2,
                val + y_max1 * 0.01,
                f'{val:.1f}',
                ha='center', va='bottom',
                fontsize=11, color='#333333', fontweight='bold'
            )
            
    apply_styling(ax1, y_max1, metric1, models, offsets1)
    plt.tight_layout()
    # fig1.savefig('Mean_Albedo_PSNR_Styled.pdf', dpi=300, bbox_inches='tight')
    plt.show()

# --- Plot 2: Angular Error ---
metric2 = 'Mean Normal Angular Error'
if metric2 in df.index:
    fig2, ax2 = plt.subplots(figsize=(8, 6))
    y_max2 = df.loc[metric2].max() * 1.25
    offsets2 = []
    
    for i, model in enumerate(models):
        offset = (i - (n_models - 1) / 2) * bar_step
        offsets2.append(offset)
        val = df.loc[metric2, model]
        
        bars = ax2.bar([offset], [val], bar_width, color=colors[i], zorder=3)
        for bar in bars:
            ax2.text(
                bar.get_x() + bar.get_width() / 2,
                val + y_max2 * 0.01,
                f'{val:.1f}',
                ha='center', va='bottom',
                fontsize=11, color='#333333', fontweight='bold'
            )
            
    apply_styling(ax2, y_max2, metric2, models, offsets2)
    plt.tight_layout()
    # fig2.savefig('Mean_Normal_Angular_Error_Styled.pdf', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
styled_df

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 1. Update typography to a clean sans-serif matching typical poster fonts
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']

# Extract the DataFrame from the Styler object
df = styled_df.data

# --- THE FIX: Filter the dataframe to only keep specific environments ---
environments_to_keep = ['city', 'courtyard', 'snow']
df = df.loc[environments_to_keep]

environments = list(df.index)
models_to_plot = ['Baseline', 'With Diffusion Priors', 'With GT']

# 2. Refine color map to match the TUM poster theme
color_map = {
    'Baseline': '#999999',              # Neutral Gray
    'Baseline + direct': '#4472C4',
    'Baseline + zncc': '#70AD47',
    'Albedo + zncc': '#9B59B6',
    
    # Add capitalized versions to match your DataFrame columns exactly
    'With GT': '#333333',               
    'With Diffusion Priors': '#0065BD', 
    
    # Keeping lowercase versions just in case
    'with GT': '#333333',               
    'with Diffusion Priors': '#0065BD', 
    'with Difusion Priors': '#0065BD'   
}

# Ensure models strictly use the mapped colors
colors = [color_map.get(model, '#333333') for model in models_to_plot]

n_models = len(models_to_plot)
bar_width = 0.25        

# --- Plot: Grouped Bar Chart by Environment ---
fig, ax = plt.subplots(figsize=(10, 6)) # Slightly narrower figure since we have fewer groups now

# X positions for the groups (environments)
x = np.arange(len(environments))

# 3. Plot grouped bars iteratively
for i, model in enumerate(models_to_plot):
    # Calculate offset so bars group nicely around the center tick
    offset = (i - n_models / 2 + 0.5) * bar_width
    
    # Extract values from the filtered dataframe
    vals = df[model]
    
    bars = ax.bar(x + offset, vals, bar_width, label=model, color=colors[i], zorder=3)
    
    # Add data labels on top of the bars
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + 0.3, 
            f'{height:.2f}', 
            ha='center', va='bottom',
            fontsize=10, color='#333333', fontweight='bold', rotation=0
        )

# 4. Apply styling
# Set the x-ticks to be the environment names
ax.set_xticks(x)
ax.set_xticklabels([str(env).capitalize() for env in environments], fontsize=12)

ax.set_ylabel('PSNR', fontsize=14)
ax.set_title('Performance by Environment Map\n(Higher is Better)', pad=15, fontsize=16)

# Set y-limits dynamically 
y_max = df[models_to_plot].max().max() * 1.15
ax.set_ylim(0, y_max)

# Grid and spine styling
ax.yaxis.grid(True, color='#e1e0d9', linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#c3c2b7')
ax.spines['bottom'].set_color('#c3c2b7')
ax.tick_params(colors='#52514e', labelsize=12)

# Add the legend below the plot 
ax.legend(
    loc='upper center', 
    bbox_to_anchor=(0.5, -0.1), 
    ncol=3, 
    frameon=False, 
    fontsize=12
)

plt.tight_layout()
plt.show()